# 🥇 ElectroFlow: Building the Gold Layer (Learning Edition)

In this notebook, you will build the **Gold Layer** of the Medallion Architecture.

## What is the Gold Layer?
The Gold Layer is the final, business-ready layer. It takes the cleaned, normalized Silver data and restructures it into a **Star Schema** optimized for BI tools and analytics.

## What is a Star Schema?
A Star Schema consists of two types of tables:
- **Dimension Tables (`dim_`)**: Descriptive attributes (WHO, WHAT, WHEN). Used for filtering and grouping.
- **Fact Tables (`fact_`)**: Measurable metrics (HOW MUCH, HOW MANY). Contains foreign keys to dimensions.

## Our Star Schema
We will build 4 tables:
1. `dim_customers` - Customer attributes
2. `dim_products` - Product catalog with prices
3. `dim_date` - Calendar dimension (generated)
4. `fact_order_items` - Transaction-grain fact table (one row per item sold)

Follow the `# TODO:` comments to complete each function! 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import date, timedelta

# Unity Catalog schema
CATALOG_SCHEMA = "dev.electroflow_pipeline."

# --- Source Tables (Silver) --- 
silver_customers_table = CATALOG_SCHEMA + "silver_customers"
silver_products_table = CATALOG_SCHEMA + "silver_products"
silver_orders_table = CATALOG_SCHEMA + "silver_orders"
silver_order_items_table = CATALOG_SCHEMA + "silver_order_items"

# --- Target Tables (Gold) ---
gold_dim_customers = CATALOG_SCHEMA + "dim_customers"
gold_dim_products = CATALOG_SCHEMA + "dim_products"
gold_dim_date = CATALOG_SCHEMA + "dim_date"
gold_fact_order_items = CATALOG_SCHEMA + "fact_order_items"

## Dimension Tables

In [0]:
def create_dim_customers():
    print("👤 Building dim_customers...")
    df = spark.table(silver_customers_table)
    df_dim = df.select("customer_id", "full_name", "gender", "email", "city", "state_province", "postal_code", "country", "country_code", "phone_number", "join_date", "_ingestion_timestamp")
    df_dim.write.format("delta").mode("overwrite").option("mergeschema", "true").saveAsTable(gold_dim_customers)
    
    return df_dim
    print("✅ dim_customers done")

In [0]:
def create_dim_products():
    print("📦 Building dim_products...")
    df = spark.table(silver_products_table)
    df_dim = spark.select("product_id", "product_name", "brand", "category", "base_price", "stock_status", "_ingestion_timestamp", "price")
    df_dim.write.format("delta").mode("overwrite").option("mergeschema", "true").saveAsTable(gold_dim_products)
    
    return df_dim
    print("✅ dim_products done")

In [0]:
def create_dim_date():
    print("📅 Building dim_date...")
    start_date = date(2024, 1, 1)
    end_date = date(2026, 12, 31)
    date_list = [{"date"}:start_date+timedelta(days=i) for i in range((end_date - start_date).days + 1)]

    df = spark.createDataFrame(date_list)

    df_dim = df\
        .withColumn("date_id", F.date_format(F.col("date"),"yyyy-MM-dd"))\
        .withColumn("year", F.year(F.col("date")))\
        .withColumn("quartter", F.quarter(F.col("date")))\
        .withColumn("month", F.month(F.col("date")))\
        .withColumn("dayofweek", F.dayofweek(F.col("date")))\
        .withColumn("is_weekend", F.when(F.dayofweek(F.col("date")).isin(1, 7), True).otherwise(False)) \
        .drop("date")

    df_dim.write.format("delta").mode("overwrite").option("mergeschema", "true").saveAsTable(gold_dim_date)

    return df_dim

    print("✅ dim_date done")

## Fact Table
This is the core of the Star Schema. Here we JOIN multiple Silver tables together to create a single, enriched fact table with calculated metrics.

In [0]:
def create_fact_order_items():
    print("📊 Building fact_order_items...")

    df_orders = spark.table(silver_orders)
    df_order_items = spark.table(silver_order_items)
    df_products = spark.table(silver_products)

    #joins
    df_joined = df_order_items.join(
        df_orders.select("order_id","customer_id","order_purchase_timestamp"),
        on = "order_id",
        how = "inner"
    )

    df_joined = df_joined.join(
        df_products.select("product_id", F.col("price").alias("unit_price")),
        on = "product_id",
        how = "inner"
    )

    df_fact = df_joined(\
        .withColumn("date_id", F.date_format(F.col("order_purchase_timestamp"), "yyyy-MM-dd")) \
        .withColumn("item_revenue", (F.col("quantity")*F.col*+("unit_price").cast("decimal(18,2)"))
      ))

    df_fact = df_fact.select(
        "order_id", "product_id", "customer_id",
        "date_id", "quantity", "unit_price", "item_revenue"
      )
    
    df_fact.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(gold_fact_order_items)
    return df_fact

    print("✅ fact_order_items done")

In [0]:
# --- Execute all Gold Layer functions ---
create_dim_customers()
create_dim_products()
create_dim_date()
create_fact_order_items()
print("🥇 Gold Layer complete!")